In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import anndata

In [ ]:
!pwd

In [ ]:
sc.set_figure_params(figsize=(5,5), frameon=False)
sc.settings.verbosity = 3

In [ ]:
projdir = "/u/project/cluo/terencew/igvf/2023_YR2/snm3C/hicluster/"

In [ ]:
indir = f'{projdir}/embedding/100K/decomp/'
npz = np.load(f'{indir}/total_decomp.npz')

In [ ]:
pcs = npz['arr_0']
pcs.shape

In [ ]:
curr_cells = pd.read_csv(f'{projdir}/csv/hicluster/imputed_100kb.csv', sep='\t', header=None, index_col=0)
curr_cells.head()

In [ ]:
!ls {projdir}/csv/pass_qc/snm3c*

In [ ]:
meta = pd.read_csv(f'{projdir}/csv/pass_qc/snm3c_hicfilter.csv',
                   sep='\t', index_col=0, header=0)
meta.drop(columns=['Unnamed: 0'], inplace=True)
meta.shape

In [ ]:
meta.head()

In [ ]:
qc_mask = curr_cells.index.isin(meta.index)
sum(qc_mask)

In [ ]:
final_cells = curr_cells[qc_mask]
final_pcs = pcs[qc_mask]
final_cells.shape, final_pcs.shape

In [ ]:
meta.index[:5], \
final_cells.index[:5]

In [ ]:
final_meta = meta.reindex(final_cells.index)
final_meta.index[:5], \
final_cells.index[:5]

In [ ]:
adata = anndata.AnnData(X=final_pcs, obs=final_meta)
adata.obsm['X_pca'] = adata.X
adata

In [ ]:
adata.obs.index.name = 'wellprefix'

In [ ]:
adata.write_h5ad(f'{projdir}/h5ad/embedding/3C_passqc_raw.h5ad')

### plot PCs

In [ ]:
sc.pl.pca(adata, components=['1,2', '3,4', '5,6', '7,8', '9,10', '11,12', '13,14', '15,16'], color='time')

In [ ]:
sc.pl.pca(adata, components=['1,2', '3,4', '5,6', '7,8', '9,10', '11,12', '13,14', '15,16'], color='line')

In [ ]:
qc_cols = ['MappingRate_Total', 'UniqueMappedReads_PostSplit',
             'contacts_final_intra', 'frac_finalcont_intragreater1kb_totintra']
### plot some QC
for col in qc_cols:
    sc.pl.pca(adata, components=['1,2', '3,4', '5,6', '7,8'], color=col)

In [ ]:
### these params will be tested in another notebook
res = 0.6
n_pcs = 10
k = 25

sc.pp.neighbors(adata, n_neighbors=k, n_pcs = n_pcs, random_state=0)
sc.tl.leiden(adata, resolution=res, random_state=0)
sc.tl.umap(adata, random_state=0)

In [ ]:
sc.pl.umap(adata)

In [ ]:
sc.pl.umap(adata, color=['time', 'line', 'sample', 'leiden'], ncols=2, wspace=0.5)

In [ ]:
sc.pl.umap(adata, color=['time', 'line', 'leiden'], wspace=0.2)

In [ ]:
qc_cols = ['contacts_final_intra', 'frac_finalcont_intragreater1kb_totintra',
           'frac_finalcont_intragreater10kb_totintra']
### plot some QC
for col in qc_cols:
    sc.pl.umap(adata, color=col)

In [ ]:
adata.write_h5ad(f'{projdir}/h5ad/embedding/3C_embedding_100K.h5ad')